# 01 - Extração Distribuída no Cluster Spark (ADLS Gen2)
**Squad 2 — Real Time for Business | Dupla 5 (Grupo 5)**  
**Integrantes:** Zaiden Emiliano Segundo Seleme *(assumindo o escopo do Grupo 5)*  
**Tabelas de Escopo:** `ecommerce_rastreamento_entregas` e `ecommerce_enderecos`  
**Branch:** `feat/squad2-dupla5-rastreamento-enderecos`  

### Objetivo:
1. Realizar a leitura distribuída e paralela de todos os micro-lotes `.parquet` depositados no container `raw/real-time-data/`.
2. Extrair os metadados nativos de arquivo (`_metadata.file_path`) para auditoria de origem.
3. Avaliar a volumetria, distribuição de schemas e consistência das chaves primárias (`id_rastreamento` e `id_endereco`).
4. Mapear o catálogo de status logísticos e a dispersão geográfica dos endereços.

In [ ]:
# Instalação de dependências no escopo da sessão Serverless
%pip install python-dotenv==1.0.1 azure-storage-file-datalake azure-identity --quiet


In [ ]:
dbutils.library.restartPython()


In [ ]:
# Carregamento seguro e resiliente das variáveis de ambiente
import os
from dotenv import load_dotenv, find_dotenv

dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env"),
        "/Workspace/Shared/estagio_ed",
        "/Workspace/Repos/estagio_ed"
    ]
    for c in candidatos:
        if os.path.exists(c):
            dotenv_path = c
            break

load_dotenv(dotenv_path, override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

assert client_id and tenant_id and client_secret, "Credenciais do Azure ADLS não encontradas no .env"
print("Credenciais do Service Principal carregadas com sucesso!")


## 1. Dicionário de Opções OAuth para o Conector Spark ABFS

In [ ]:
# Configurações OAuth com FQDN do Service Principal injetadas a cada chamada do Spark
adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

import io
import pyarrow.parquet as pq
import pyarrow as pa
from azure.storage.filedatalake import DataLakeServiceClient
from azure.identity import ClientSecretCredential

schema_rastreamento_pa = pa.schema([
    ('id_rastreamento', pa.int64()),
    ('id_pedido_ecommerce', pa.int64()),
    ('codigo_rastreio', pa.string()),
    ('id_transportadora', pa.int64()),
    ('nome_transportadora', pa.string()),
    ('status_entrega', pa.string()),
    ('dt_evento', pa.timestamp('us')),
    ('observacao', pa.string()),
])

schema_enderecos_pa = pa.schema([
    ('id_endereco', pa.int64()),
    ('id_cliente', pa.int64()),
    ('logradouro', pa.string()),
    ('numero', pa.int64()),
    ('complemento', pa.string()),
    ('bairro', pa.string()),
    ('cep', pa.string()),
    ('cidade', pa.string()),
    ('estado', pa.string()),
    ('latitude', pa.float64()),
    ('longitude', pa.float64()),
    ('apelido', pa.string()),
    ('is_principal', pa.bool_()),
])

def ler_parquet_raw_resiliente(nome_arquivo_alvo, schema_pa):
    """
    Lê os micro-lotes do container raw contornando limitações do Parquet Vectorized Reader
    no Databricks Serverless (trata INT32 Null em campos string e TIMESTAMP_NANOS).
    """
    cred = ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret)
    service = DataLakeServiceClient(account_url=f"https://{storage_account}.dfs.core.windows.net", credential=cred)
    fs = service.get_file_system_client("raw")
    
    paths = fs.get_paths(path="real-time-data")
    arquivos = [p.name for p in paths if p.name.endswith(nome_arquivo_alvo)]
    
    tables = []
    for p in arquivos:
        fc = fs.get_file_client(p)
        data = fc.download_file().readall()
        t = pq.read_table(io.BytesIO(data))
        t_casted = t.cast(schema_pa)
        full_path = f"abfss://raw@{storage_account}.dfs.core.windows.net/{p}"
        t_with_meta = t_casted.append_column("bronze_source_file", pa.array([full_path] * len(t_casted)))
        tables.append(t_with_meta)
        
    if not tables:
        return None
    combined = pa.concat_tables(tables)
    return spark.createDataFrame(combined.to_pandas())


## 2. Leitura Paralelizada e Distribuída de `ecommerce_rastreamento_entregas`

In [ ]:
from pyspark.sql.functions import col, countDistinct, count

# Leitura resiliente de rastreamento com resolução de TIMESTAMP_NANOS
df_raw_rastreamento = ler_parquet_raw_resiliente("ecommerce_rastreamento.parquet", schema_rastreamento_pa)

total_rastreamento = df_raw_rastreamento.count()
arquivos_rastreamento = df_raw_rastreamento.select(countDistinct("bronze_source_file")).first()[0]

print(f"=== Extração de Rastreamento de Entregas ===")
print(f"Total de registros lidos: {total_rastreamento}")
print(f"Total de arquivos parquet particionados: {arquivos_rastreamento}")

print("\nSchema inferido de ecommerce_rastreamento_entregas:")
df_raw_rastreamento.printSchema()

display(df_raw_rastreamento.limit(5))


## 3. Leitura Paralelizada e Distribuída de `ecommerce_enderecos`

In [ ]:
# Leitura resiliente de endereços com resolução de INT32 Null
df_raw_enderecos = ler_parquet_raw_resiliente("ecommerce_enderecos.parquet", schema_enderecos_pa)

total_enderecos = df_raw_enderecos.count()
arquivos_enderecos = df_raw_enderecos.select(countDistinct("bronze_source_file")).first()[0]

print(f"=== Extração de Endereços ===")
print(f"Total de registros lidos: {total_enderecos}")
print(f"Total de arquivos parquet particionados: {arquivos_enderecos}")

print("\nSchema inferido de ecommerce_enderecos:")
df_raw_enderecos.printSchema()

display(df_raw_enderecos.limit(5))


## 4. Análise Exploratória e Validação Inicial de Qualidade

In [ ]:
# 1. Unicidade das chaves primárias no RAW
rastreio_unicos = df_raw_rastreamento.select(countDistinct("id_rastreamento")).first()[0]
enderecos_unicos = df_raw_enderecos.select(countDistinct("id_endereco")).first()[0]

print(f"Unicidade de Chaves:")
print(f"  id_rastreamento únicos: {rastreio_unicos} / {total_rastreamento} registros")
print(f"  id_endereco únicos:     {enderecos_unicos} / {total_enderecos} registros")

# 2. Distribuição dos Status Logísticos de Entrega
print("\nDistribuição de Status de Entrega:")
df_raw_rastreamento.groupBy("status_entrega").agg(count("*").alias("total")).orderBy(col("total").desc()).show(truncate=False)

# 3. Distribuição Geográfica de Endereços por Estado (Top 10)
print("\nDistribuição Geográfica dos Endereços por Estado:")
df_raw_enderecos.groupBy("estado").agg(count("*").alias("total")).orderBy(col("total").desc()).show(10, truncate=False)

print("Extração distribuída e auditoria exploratória concluídas com sucesso!")
